# Module 4: Failure Lab - Redis

## ⚠️ CRITICAL SAFETY WARNING

**THIS NOTEBOOK WILL MODIFY YOUR ENVIRONMENT:**
- **Modifies Kubernetes secrets** (breaks Redis password)
- **Causes service disruptions** (intermittent ingestion, worker backlog)
- **Requires remediation** to restore functionality

**REQUIREMENTS:**
- ✅ **TEST/NON-PRODUCTION environment ONLY**
- ✅ **MODULE4_SAFE_ENVIRONMENT=true** must be set in .env
- ✅ **Baseline diagnostics collected** (run `01_diagnostics_baseline.ipynb` first)
- ✅ **Backup/restore plan** available

**DO NOT RUN THIS LAB AGAINST PRODUCTION SYSTEMS.**

## Overview

**This lab teaches you how to debug Redis connectivity failures in LangSmith.**

Redis is LangSmith's **cache and job queue**. It handles:
- Job queue for asynchronous trace processing
- Caching for frequently accessed data
- Rate limiting and session management
- Worker coordination

**When Redis fails, you'll see:**
- Intermittent ingestion issues
- Latency spikes and retries
- Worker backlog (jobs piling up)
- Traces may be delayed or missing

**Learning Objectives:**
1. Understand how Redis failures manifest
2. Practice collecting diagnostics for cache/queue issues
3. Learn to identify connection vs. credential vs. network issues
4. Practice safe remediation

**Estimated time:** 30-45 minutes

**⚠️ Important:** 
- Run `01_diagnostics_baseline.ipynb` BEFORE starting this lab!
- Complete safety check in `00_setup_or_resume_environment.ipynb` first!


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path
possible_paths = [
    Path.cwd().parent,
    Path.cwd(),
    Path.cwd() / "notebooks",
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap
from shared._validation import ok, warn

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## ⚠️ CRITICAL: Environment Safety Verification

**Before proceeding, verify you're in a TEST/NON-PRODUCTION environment and understand what will be modified.**


In [ ]:
# CRITICAL SAFETY CHECK: Verify environment is safe for failure injection
from shared._cloud_helpers import get_cloud_provider, get_region, get_identity
from shared._validation import ok, warn, fail
import os

provider = get_cloud_provider()
region = get_region()
identity = get_identity()

print("=" * 70)
print("⚠️  CRITICAL SAFETY CHECK - REDIS FAILURE LAB")
print("=" * 70)

# Show environment details prominently
provider_display = provider.upper()
print(f"\n### Current Environment Configuration")
print(f"Cloud Provider: {provider_display}")
print(f"Region: {region}")

if provider == "aws":
    account_id = identity.get('Account', 'N/A')
    user_arn = identity.get('Arn', 'N/A')
    print(f"Account ID: {account_id}")
    print(f"User ARN: {user_arn}")
elif provider == "azure":
    subscription_id = identity.get("SubscriptionId") or identity.get("Account", "N/A")
    subscription_name = identity.get("SubscriptionName", "N/A")
    print(f"Subscription ID: {subscription_id}")
    print(f"Subscription Name: {subscription_name}")

# Show all relevant environment variables
print(f"\n### Environment Variables (VERIFY THESE ARE CORRECT)")
print(f"NAMESPACE: {os.environ.get('NAMESPACE', 'NOT SET')}")
print(f"CLUSTER_NAME: {os.environ.get('CLUSTER_NAME', 'NOT SET')}")
print(f"HELM_RELEASE: {os.environ.get('HELM_RELEASE', 'langsmith')}")
print(f"LANGSMITH_DOMAIN: {os.environ.get('LANGSMITH_DOMAIN', 'NOT SET')}")

print("\n" + "=" * 70)
print("⚠️  WHAT THIS LAB WILL DO:")
print("=" * 70)
print("\nThis failure lab will:")
print("  1. Find the Redis secret in your namespace")
print("  2. BACKUP the original secret (saved to artifacts)")
print("  3. MODIFY the secret to set an INVALID password")
print("  4. Apply the modified secret (breaks Redis connectivity)")
print("  5. Cause intermittent ingestion issues and worker backlog")
print("  6. Require remediation to restore (restore original secret)")
print("\n" + "=" * 70)

# Check for Module 4 safety flag
module4_safe = os.environ.get("MODULE4_SAFE_ENVIRONMENT", "").lower()
if module4_safe not in ["true", "yes", "1"]:
    fail("MODULE4_SAFE_ENVIRONMENT flag is NOT set")
    print("\n❌ SAFETY CHECK FAILED - Cannot proceed")
    print("\nTo run this failure lab, you MUST:")
    print("  1. Verify this is a TEST/NON-PRODUCTION environment")
    print("  2. Set MODULE4_SAFE_ENVIRONMENT=true in your .env file")
    print("  3. Complete safety check in 00_setup_or_resume_environment.ipynb")
    print("  4. Re-run this cell to confirm")
    print("\nThis flag is REQUIRED to prevent accidental execution in production.")
    raise RuntimeError("MODULE4_SAFE_ENVIRONMENT not set. Required for failure labs.")

ok("MODULE4_SAFE_ENVIRONMENT flag is set")
print("\n✅ Safety check passed - environment marked as safe for failure injection")
print("\n⚠️  REMINDER: This lab will break Redis connectivity.")
print("   Ensure you understand the remediation steps before proceeding.")
print("   Original secret will be backed up automatically.")

print("\n" + "=" * 70)
print("✅ Environment verified - ready for Redis failure lab")
print("=" * 70)


## 1. Configuration & Prerequisites

Load configuration and verify prerequisites.


In [ ]:
import os
from shared._validation import require_env

required_vars = ["NAMESPACE", "CLUSTER_NAME"]
config = require_env(*required_vars)
config["HELM_RELEASE"] = os.environ.get("HELM_RELEASE", "langsmith")

namespace = config["NAMESPACE"]

print(f"Namespace: {namespace}")
print(f"Helm Release: {config['HELM_RELEASE']}")

ok("Configuration loaded")


## 2. What This Service Does for LangSmith

Redis is LangSmith's **cache and job queue**. It handles:

- **Job queue for asynchronous processing:**
  - Workers pull trace processing jobs from Redis
  - Jobs are queued when traces arrive via API
  - Queue backlog indicates processing delays

- **Caching:**
  - Frequently accessed data (project metadata, user info)
  - Reduces load on PostgreSQL
  - Improves response times

- **Rate limiting and session management:**
  - API rate limiting
  - Session storage (if configured)

- **Worker coordination:**
  - Distributed locking
  - Task distribution

**Why it matters:**
- Without Redis, workers can't process traces
- Job queue fills up, causing delays
- Cache misses increase load on PostgreSQL
- Ingestion becomes unreliable

**How LangSmith connects:**
- Connection string stored in Kubernetes Secrets
- Workers connect to Redis to pull jobs
- API servers use Redis for caching


## 3. Expected Symptoms When Redis Fails

**What you'll see:**

1. **Intermittent ingestion issues:**
   - Some traces process, others don't
   - Inconsistent behavior (works sometimes, fails other times)
   - Retries visible in logs

2. **Latency spikes:**
   - API responses slow down
   - Worker processing delays
   - Timeout errors

3. **Worker backlog:**
   - Jobs piling up in queue
   - Workers unable to pull new jobs
   - Queue length increasing

4. **Log patterns:**
   - Connection timeout errors
   - "connection refused" or "connection reset"
   - "NOAUTH Authentication required" (if password wrong)
   - Retry attempts in worker logs
   - Cache miss patterns

**Timeline:**
- Symptoms may be intermittent (connection pool retries)
- Worker backlog builds over time
- Cache misses cause cascading delays
- Full failure if connection pool exhausted


## 4. Failure Injection Options

**Choose ONE level to practice with. Level 1 is subtle, Level 2 is more obvious.**

### Level 1: Subtle Failure (Recommended for first run)

**Option A: Wrong Redis Password**
- Modify the Redis password in the Kubernetes Secret
- Symptoms: Authentication failures, connection refused, intermittent failures

**Option B: Block Egress to Redis Endpoint**
- Apply NetworkPolicy blocking egress to Redis (if NetworkPolicy supported)
- Symptoms: Connection timeout, no route to host, intermittent failures

### Level 2: Obvious Failure

**Option C: Wrong Redis Host/Endpoint**
- Point connection string to non-existent host
- Symptoms: Connection timeout, DNS resolution failures, immediate failures

**⚠️ Safety:** All injections are reversible. We'll save the original secret before modifying it.


## 5. Do the Drill - Step 1: Confirm Baseline

**Before injecting any failure, verify your baseline is healthy.**

💡 **If you haven't run `01_diagnostics_baseline.ipynb` yet, do that first!**


In [ ]:
from shared._shell import run
import json

print("### Quick Baseline Check\n")

# Check pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    healthy = sum(1 for p in pods.get("items", [])
                  if p.get("status", {}).get("phase") == "Running")
    total = len(pods.get("items", []))
    print(f"Pods: {healthy}/{total} running")
    
    if healthy == total and total > 0:
        ok("Baseline looks healthy")
    else:
        warn("Some pods are not running - check baseline first")
else:
    warn("Could not check pod status")

# Check for Redis secret
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

redis_secrets = []
if result.returncode == 0:
    secrets = json.loads(result.stdout)
    for secret in secrets.get("items", []):
        name = secret.get("metadata", {}).get("name", "")
        if "redis" in name.lower() or "cache" in name.lower():
            redis_secrets.append(name)

if redis_secrets:
    ok(f"Found {len(redis_secrets)} Redis-related secret(s)")
    for secret_name in redis_secrets:
        print(f"   - {secret_name}")
else:
    warn("No Redis secrets found")
    print("   💡 Redis connection may be configured differently")


## 6. Do the Drill - Step 2: Apply Failure Injection

**⚠️ WARNING: This will modify your LangSmith deployment. Make sure you're in a test environment!**

Choose your failure injection method below. We'll use **Option A (Wrong Password)** as the default example.


In [ ]:
# FAILURE INJECTION: Wrong Redis Password
# This cell modifies the Redis password secret to an invalid value

import base64
import yaml
from datetime import datetime

# Find Redis secret (look for common names)
result = run(
    ["kubectl", "get", "secrets", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

redis_secret_name = None
if result.returncode == 0:
    secrets = json.loads(result.stdout)
    for secret in secrets.get("items", []):
        name = secret.get("metadata", {}).get("name", "")
        # Common patterns: redis, cache
        if any(keyword in name.lower() for keyword in ["redis", "cache"]):
            # Check if it has password-related keys
            data = secret.get("data", {})
            if any(key in data for key in ["password", "REDIS_PASSWORD", "CACHE_PASSWORD"]):
                redis_secret_name = name
                break

if not redis_secret_name:
    raise RuntimeError("❌ Could not find Redis secret. Check your deployment configuration.")

print(f"Found Redis secret: {redis_secret_name}")

# Get current secret
result = run(
    ["kubectl", "get", "secret", redis_secret_name, "-n", namespace, "-o", "yaml"],
    check=True,
    stream=False
)

# Save original secret for restoration
backup_file = artifacts_dir / "module-4" / f"redis-secret-backup-{datetime.now().strftime('%Y%m%d-%H%M%S')}.yaml"
backup_file.parent.mkdir(parents=True, exist_ok=True)
with open(backup_file, "w") as f:
    f.write(result.stdout)

ok(f"Backed up original secret to: {backup_file.name}")

# Parse YAML and modify password
secret_data = yaml.safe_load(result.stdout)
if "data" not in secret_data:
    raise RuntimeError("Secret has no data section")

# Find password key (could be password, REDIS_PASSWORD, CACHE_PASSWORD, etc.)
password_key = None
for key in ["password", "REDIS_PASSWORD", "CACHE_PASSWORD", "redis-password"]:
    if key in secret_data["data"]:
        password_key = key
        break

if not password_key:
    raise RuntimeError("Could not find password key in secret")

# Set invalid password
invalid_password = "INVALID_PASSWORD_12345"
invalid_password_b64 = base64.b64encode(invalid_password.encode()).decode()

# Modify secret
secret_data["data"][password_key] = invalid_password_b64

# Save modified secret to temp file
temp_secret_file = artifacts_dir / "module-4" / "redis-secret-modified.yaml"
with open(temp_secret_file, "w") as f:
    yaml.dump(secret_data, f)

print(f"\n⚠️  READY TO APPLY FAILURE INJECTION")
print(f"   This will set an invalid password in secret: {redis_secret_name}")
print(f"   Modified secret saved to: {temp_secret_file.name}")
print(f"\n   To apply, uncomment and run the next cell.")


In [ ]:
# UNCOMMENT TO APPLY FAILURE INJECTION
# 
# result = run(
#     ["kubectl", "apply", "-f", str(temp_secret_file)],
#     check=True,
#     stream=True
# )
# 
# ok("Failure injection applied - Redis password is now invalid")
# print("\n💡 Pods will need to restart to pick up the new secret.")
# print("   This may take 1-2 minutes. Watch for pod restarts:")
# print(f"   kubectl get pods -n {namespace} -w")
# 
# # Wait a moment for changes to propagate
# import time
# print("\nWaiting 30 seconds for changes to propagate...")
# time.sleep(30)


## 8. Do the Drill - Step 3: Observe Symptoms

**Now that the failure is injected, observe how it manifests.**

Check:
1. Worker pod logs for Redis connection errors
2. Queue backlog (if visible)
3. Worker retry patterns
4. Latency in API responses


In [ ]:
from datetime import datetime

# Create incident directory for diagnostics
incident_dir = artifacts_dir / "module-4" / f"redis-failure-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
incident_dir.mkdir(parents=True, exist_ok=True)

print(f"### Collecting Failure Diagnostics\n")
print(f"Saving to: {incident_dir}\n")

# 1. Check pod status
print("1. Checking pod status...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "wide"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(incident_dir / "pods-status.txt", "w") as f:
        f.write(result.stdout)
    print(result.stdout)
    
    # Check for restarts
    lines = result.stdout.split("\n")
    restarts = [l for l in lines if "RESTARTS" in l or (l and not l.startswith("NAME"))]
    if restarts:
        print("\n   Pod restart counts:")
        for line in restarts[1:]:  # Skip header
            if line.strip():
                parts = line.split()
                if len(parts) > 3:
                    print(f"   {parts[0]}: {parts[3]} restarts")

# 2. Check recent events
print("\n2. Checking recent events...")
result = run(
    ["kubectl", "get", "events", "-n", namespace, "--sort-by='.lastTimestamp'", "--field-selector=type!=Normal"],
    check=False,
    stream=False
)

if result.returncode == 0:
    with open(incident_dir / "events.txt", "w") as f:
        f.write(result.stdout)
    if result.stdout.strip():
        print("   Recent warning/error events:")
        for line in result.stdout.split("\n")[-5:]:
            if line.strip():
                print(f"   {line}")

# 3. Check worker pod logs for Redis errors
print("\n3. Checking worker pod logs for Redis errors...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-l", "app=langsmith-worker", "-o", "jsonpath='{.items[0].metadata.name}'"],
    check=False,
    stream=False
)

worker_pod = result.stdout.strip().strip("'\"")
if worker_pod:
    result = run(
        ["kubectl", "logs", "-n", namespace, worker_pod, "--tail=50"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        logs_file = incident_dir / f"worker-pod-{worker_pod}-logs.txt"
        with open(logs_file, "w") as f:
            f.write(result.stdout)
        
        # Look for Redis-related errors
        error_keywords = ["redis", "cache", "connection", "timeout", "refused", "authentication", "NOAUTH", "retry"]
        error_lines = [l for l in result.stdout.split("\n") 
                      if any(kw in l.lower() for kw in error_keywords)]
        
        if error_lines:
            print("   Found Redis-related errors:")
            for line in error_lines[-5:]:
                print(f"   {line}")
        else:
            print("   No obvious Redis errors in recent logs")
else:
    warn("Could not find worker pod")

ok(f"Diagnostics saved to: {incident_dir}")


## 9. Do the Drill - Step 4: Run Canonical Diagnostics Script

**This is critical - Support will ask for this bundle.**


In [ ]:
import urllib.request

print("### Running Canonical Diagnostics Script\n")

script_url = "https://raw.githubusercontent.com/langchain-ai/helm/main/charts/langsmith/scripts/get_k8s_debugging_info.sh"
script_path = incident_dir / "get_k8s_debugging_info.sh"

try:
    urllib.request.urlretrieve(script_url, script_path)
    script_path.chmod(0o755)
    
    print(f"Running diagnostics script for namespace: {namespace}")
    result = run(
        [str(script_path), namespace],
        check=False,
        stream=True
    )
    
    if result.returncode == 0:
        ok("Diagnostics script completed")
        
        # Find and move tarball
        for file in incident_dir.parent.iterdir():
            if file.name.startswith("langsmith-debug-") and file.suffix == ".tar.gz":
                target_path = incident_dir / file.name
                file.rename(target_path)
                ok(f"Diagnostics bundle: {target_path.name}")
                break
    else:
        warn("Diagnostics script had errors (check output above)")
        
except Exception as e:
    warn(f"Could not run diagnostics script: {e}")
    print("   💡 You can run it manually:")
    print(f"      curl -O {script_url}")
    print(f"      chmod +x get_k8s_debugging_info.sh")
    print(f"      ./get_k8s_debugging_info.sh {namespace}")


## 10. Do the Drill - Step 5: Guided Triage

**Where to look first for Redis issues:**


In [ ]:
print("### Guided Triage Steps\n")

print("1. Check worker pod logs for Redis connection errors:")
print(f"   kubectl logs -n {namespace} <worker-pod-name> | grep -i 'redis\\|cache\\|connection'")
print()

print("2. Verify secret exists and has correct keys:")
print(f"   kubectl get secret {redis_secret_name} -n {namespace} -o yaml")
print("   (Don't print the actual values - they're base64 encoded)")
print()

print("3. Check for worker pod restarts (indicates connection failures):")
print(f"   kubectl get pods -n {namespace} -l app=langsmith-worker")
print()

print("4. Test Redis connectivity from a pod (if possible):")
print("   kubectl run -it --rm debug --image=redis:7 --restart=Never -- \\")
print("     redis-cli -h <redis-host> -p <port> -a <password> ping")
print()

print("5. Check events for connection/authentication errors:")
print(f"   kubectl get events -n {namespace} --sort-by='.lastTimestamp' | grep -i 'error\\|fail'")
print()

# Check what we can automatically
print("\n### Automatic Checks\n")

# Check secret still exists
result = run(
    ["kubectl", "get", "secret", redis_secret_name, "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    ok(f"Secret '{redis_secret_name}' still exists")
    secret_data = json.loads(result.stdout)
    keys = list(secret_data.get("data", {}).keys())
    print(f"   Secret keys: {', '.join(keys)}")
else:
    warn(f"Secret '{redis_secret_name}' not found!")

# Check for pods with Redis connection env vars
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    redis_related_pods = []
    for pod in pods.get("items", []):
        name = pod.get("metadata", {}).get("name", "")
        containers = pod.get("spec", {}).get("containers", [])
        for container in containers:
            env = container.get("env", [])
            redis_env = [e for e in env if any(kw in e.get("name", "").upper() 
                                              for kw in ["REDIS", "CACHE"])]
            if redis_env:
                redis_related_pods.append(name)
                break
    
    if redis_related_pods:
        print(f"\n   Pods with Redis environment variables:")
        for pod_name in set(redis_related_pods):
            print(f"   - {pod_name}")


## 11. Do the Drill - Step 6: Remediation

**Restore the original secret to fix the issue.**


In [ ]:
# REMEDIATION: Restore original secret
# UNCOMMENT TO RESTORE

# if backup_file.exists():
#     print(f"Restoring original secret from: {backup_file.name}")
#     result = run(
#         ["kubectl", "apply", "-f", str(backup_file)],
#         check=True,
#         stream=True
#     )
#     
#     ok("Original secret restored")
#     print("\n💡 Pods will restart to pick up the correct secret.")
#     print("   This may take 1-2 minutes. Monitor pod status:")
#     print(f"   kubectl get pods -n {namespace} -w")
#     
#     import time
#     print("\nWaiting 60 seconds for pods to restart...")
#     time.sleep(60)
# else:
#     warn(f"Backup file not found: {backup_file}")
#     print("   💡 You may need to manually restore the secret")

print("⚠️  To restore, uncomment the code above and run this cell.")
print(f"   Backup file: {backup_file.name}")


## 12. Do the Drill - Step 7: Confirm Recovery

**Verify that everything is working again.**


In [ ]:
print("### Verifying Recovery\n")

# Check pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    pods = json.loads(result.stdout)
    running = sum(1 for p in pods.get("items", [])
                  if p.get("status", {}).get("phase") == "Running")
    total = len(pods.get("items", []))
    
    if running == total and total > 0:
        ok(f"All {total} pod(s) are running")
    else:
        warn(f"Only {running}/{total} pod(s) running")
        print("   💡 Wait a bit longer for pods to fully recover")

# Check for recent errors in worker logs
print("\nChecking for recent errors in worker logs...")
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-l", "app=langsmith-worker", "-o", "jsonpath='{.items[0].metadata.name}'"],
    check=False,
    stream=False
)

worker_pod = result.stdout.strip().strip("'\"")
if worker_pod:
    result = run(
        ["kubectl", "logs", "-n", namespace, worker_pod, "--tail=20"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        error_keywords = ["error", "fail", "redis", "cache", "connection"]
        recent_errors = [l for l in result.stdout.split("\n") 
                        if any(kw in l.lower() for kw in error_keywords)]
        
        if recent_errors:
            warn("Still seeing some errors in logs:")
            for line in recent_errors[-3:]:
                print(f"   {line}")
        else:
            ok("No recent errors in worker logs")

ok("Recovery verification complete")


## 13. What Support Will Ask For

**When escalating a Redis issue, Support will need:**

1. **Diagnostics bundle** (canonical script output) ✅ Collected above
2. **Redis connection details:**
   - Host/endpoint (redacted)
   - Port
   - Password (redacted)
   - Whether using SSL/TLS
3. **Error messages from logs:**
   - Full error text (not just "connection failed")
   - Timestamps of first occurrence
   - Retry patterns
4. **Recent changes:**
   - Secret rotations
   - Network policy changes
   - Redis configuration changes
5. **Queue status (if accessible):**
   - Queue length
   - Worker processing rate
   - Backlog growth rate
6. **Redis health (if accessible):**
   - Redis version
   - Memory usage
   - Connection count
   - Slow queries

**Evidence collected in this lab:**
- ✅ Diagnostics bundle
- ✅ Worker pod logs with Redis errors
- ✅ Events showing failures
- ✅ Secret configuration (structure, not values)

**Additional evidence to gather (if escalating):**
- Redis endpoint connectivity test
- Queue metrics (if available)
- Redis logs (if accessible via cloud provider)


## 14. Lessons Learned

**Key takeaways from this lab:**

1. **Redis failures can be intermittent** - Connection pool retries may mask issues
2. **Worker logs are critical** - Redis errors appear in worker pod logs
3. **Queue backlog is a symptom** - Jobs pile up when workers can't connect
4. **Secrets matter** - Wrong credentials cause authentication failures
5. **Baseline is critical** - You need "before" to compare to "after"

**Common mistakes to avoid:**
- ❌ Ignoring intermittent failures (they indicate connection issues)
- ❌ Not checking worker logs (API logs may not show Redis errors)
- ❌ Not monitoring queue length (backlog indicates processing delays)
- ❌ Not testing Redis connectivity independently

**Next steps:**
- Practice with other failure injection methods (Level 2)
- Try the ClickHouse or Blob Storage failure labs
- Review the [First 10 Minutes Checklist](../shared/incident_first_10_minutes.md)
